<a href="https://colab.research.google.com/github/ajcollins16/DSST289_Fall2026_AJC/blob/main/Notebooks/Notebook14_StatInference2_SV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 14 — Statistical Inference Two

**Main Objective**

1. Independent Samples (Two-Sample) t-Test
2. One-Way (Between Subjects) ANOVA
3. Chi-Squared Test (of Independence)

## Setup

Please run the setup code below

In [ ]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds-py/refs/heads/main/funs.py

from funs import *

In [ ]:
import numpy as np
import polars as pl

from plotnine import *
from polars import col as c
theme_set(theme_minimal())

path = "https://raw.githubusercontent.com/ajcollins16/DSST289_Fall2026_AJC/refs/heads/main/"

In [ ]:
import string
lower_set = set(string.ascii_lowercase)
keylogs = pl.read_csv(path + "Data/keylogs-1774817569168.csv")

In [ ]:
# Showing what this dataset looks like.
keylogs

## Mean Comparisons

We will be using the the `keylogs` dataset today. Each row of this dataset represents a single recorded interaction from the keyboard or mouse. For example the pressing of a specific key, the releasing of that key, moving the mouse, clicking the mouse. A single typed character will therefore usually produce several rows: a `down` event, and `input` event, and an `up` event. So the columns in the dataset are as follows:

* `time` --> The elapsed time of the event, measured in milliseconds from roughly the beginning of that recording.
* `type` --> what kind of recording event happened: `down`, `up`, `input`, `mouse`, and `click`.
* `key` --> The character or keyboard action associated with the event. So things like `a`, `I`, `Backspace`, `Shift`, and `ArrowLeft`.
* `key_code` --> A more technical description of they kery and input actions.
* `alt_key` --> A boolean variable indicating whether the Alt/Option key was being held down during the event.
* `cltr_key` --> A boolean variable indicating whether the control key was being held down during the event.
* `meta_key` --> A boolean variable for if a operating system specific key was being held down (like the command key on mac or windows key on windows) during the event.
* `shift_key` --> A boolean variable for if a shift key was being held down during the event.
* `is_repeat` --> A boolean variable to inidcate whether the event was generating because a key was held down long enough to repeat automatically.
* `range_start` --> For keyboard and text-input events, this is the starting position of the cursor or text selection within the document.
* `rang_end` --> For keyboard and text-input events, this is the ending position of the cursor or text selection within the document.

We will be cleaning this dataset first and then using it to assess some specific mean comparisons techniques that use the independent samples t-test, one-way (between subjects) ANOVA, and the chi-squared test (of independence).

### Questions

1. Start by looking at the first ten rows of the data that correspond to the `down` and `up` types (i.e., `types` column).  

2. Before moving forward, look at the `lower_set` variable that was created above.

2. What is this variable?

**Answer (Q2):**

3. Now, we want to figure out how long each key was held down for. To achieve this, we first need to pair each `down` and `up` events. Specifically, we need to grab the `time` value for the `up` type and place it with the `down` type in a new variable (let's call it `finish_time`). Below is the method to accomplish this (since we have not learned this at this point.

In [ ]:
(
    keylogs
    .filter(c.type.is_in(["down", "up"]))
    .with_columns(
        finish_time = c.time.shift(-1).over(c.key)
    )
)

3. For some additional context. This code is basically keeping only the `up` and `down` button presses. It then creates a new variable called `finish_time`, where `.shift(-1)` grabs the time of the next event for that same key which is when that key was released. So it matches the key's together so things don't get awkward.

4. Now, clean this up into a usable dataset. First, start from the previous code and keep only the `down` rows (since we do not need the `up` rows anymore since they will be empty/irrelevant).

5. Next, continuing to add to the above, remove any rows where `finish_time` is still `null` (implying keys they were never "released."

6. Finally, again adding to the above, compute a `duration` variable which subtracts the `finish_time` from the `time` to calculate the time it took for the button to be pressed and released. Importantly, call this whole sequence of cleaning the `clean_keylogs_dat`.

7. Now, let's run some tests. First, assess whether two specific keys have different hold durations. Specifically, if there is a difference between the "Shift" and "." button presses. As a reminder, you will use the wrapper function `DSStatsmodels.ttest2`; making sure to use the correct "formula."

8. Let's visualize the holding duration across all lowercase letter keys. Filter the cleaned dataset by the `lower_set` and create a boxplot with the `key` on the x-axis (reordered by `duration`) and `duration` on the y-axis.

8. From the visual, what keys tend to be held down the longest?

**Answer (Q8):**

9. Let's now extend Q7, using the `DSStatsmodels.anova` wrapper to run an ANOVA test on the lowercase letter keys, testing whether the mean hold duration differs across keys. Filter to `lower_set` as before.

10. Continuing, let us investigate whether pressing Backspace is associated with the previous key being a space (which might indicate deleting a whole word). Create two new columns: `is_backspace` (whether the current key is "Backspace") and `is_space` (whether the *previous* key was a space — hint: use `.shift(1)` to look back one row). Then group by both columns, count the rows in each group, and pivot the result into a contingency table.

11. Now, formally test the association from the contingency table we created above using the chi-squared test of independence. Use the `DSStatsmodels.chi2` wrapper with the formula `"is_backspace ~ is_space"` on the `key_data` dataset (with the same two new columns computed inline).